In [1]:
# Housekeeping 
from pathlib import Path
import pickle
import copy

# Data Processing
import pandas as pd
import numpy as np

# Geospatial
import geopandas as gpd
from shapely.geometry import Point

# Math and Stats
from scipy.stats import linregress

# Plotting
import matplotlib.pyplot as plt 

# Custom Modules
import src
import src.importData as importData
import src.geoSlicing as geoSlicing
import src.makePlots as makePlots

import importlib
importlib.reload(src.importData)
importlib.reload(src.timeConversions)
importlib.reload(src.makePlots)
importlib.reload(src.geoSlicing)


<module 'src.geoSlicing' from 'e:\\NPP\\STAQS\\src\\geoSlicing.py'>

#### Importing Data

##### Import TOLNet and Sonde Data

In [2]:
data_dir = Path('./data')


data = importData.main_import(data_dir)
tolnet = data['tolnet'].copy()
sondes = data['sondes'].copy()

##### Process ASOS Data

In [3]:

if (data_dir / 'processed_asos_data.pkl').is_file():
    with open(data_dir / 'processed_asos_data.pkl', 'rb') as f:
        asos, asos_geometry = pickle.load(f)
else:
    asos = []; asos_geometry = []
    for file in (data_dir / "Surface").glob('*.csv'):
        temp = pd.read_csv(file, low_memory=False, header=0)
        result = importData.convert_asos(temp)
        asos.append(result["data"])
        asos_geometry.append(result["geometry"])
        
    asos = pd.concat(asos)
    asos_geometry = pd.concat(asos_geometry).drop_duplicates(keep='first').reset_index(drop=True)

    with open(data_dir / 'processed_asos_data.pkl', 'wb') as f:
        pickle.dump((asos, asos_geometry), f)

##### Process EPA AirNow Data

In [19]:
# Process EPA AirNow Data
if (data_dir / 'processed_epa_data.pkl').is_file():
    with open(data_dir / 'processed_epa_data.pkl', 'rb') as f:
        airnow_data, airnow_metadata = pickle.load(f)
else:
    from atmoz.surface.AirNow import AirNow

    airnow = AirNow()
    airnow_data = airnow.import_data(
        date_start='2023-07-23', 
        date_end="2023-08-19", 
        BBOX=["-74.50094662263689", "40.24906753646499", "-71.08908282449097", "42.030950926289144"], 
        parameters=["OZONE", "NO2", "PM25", "PM10", "SO2", "CO"]
        )

    surface = {} 
    for key in airnow_data[0]: 

        if key not in surface:
            surface[key] = []
        
        for sub_key in airnow_data[0][key]:
            temp = airnow_data[1][airnow_data[1]["id"] == sub_key]
            geometry = [Point(temp["Longitude"], temp["Latitude"])] * len(airnow_data[0][key][sub_key])
            new = airnow_data[0][key][sub_key].copy()
            new["geometry"] = geometry
            surface[key].append(new)
        
        surface[key] = gpd.GeoDataFrame(
            pd.concat(surface[key]), 
            geometry="geometry"
            )
    
    with open(data_dir / 'processed_epa_data.pkl', 'wb') as f:
        pickle.dump((surface, airnow_data[1]), f)

##### Create TOLNet vs Sonde Comparison

In [5]:
# Create TOLNet vs Sonde Comparison
if (data_dir / "comparison.pkl").is_file(): 
    with open(data_dir / "comparison.pkl", "rb") as f:
        comparison = pickle.load(f)
else:
    comparison = {} 
    for tolnet_key in tolnet.keys(): 
        for sonde_key in sondes.index.get_level_values('filename').unique():
            sondes_masked = geoSlicing._add_distance_from_point(sondes.loc[sonde_key, :], location=tolnet[tolnet_key].geometry.unique()[0])
            temp = makePlots._match_time(sondes_masked[sondes_masked["distance_km"] <= 10].drop(columns="geometry"), tolnet[tolnet_key].drop(columns="geometry"))
            if not temp["left_df"].empty and not temp["right_df"].empty: 
                comparison[(tolnet_key, sonde_key)] = temp

    with open(data_dir / "comparison.pkl", "wb") as f:
        pickle.dump(comparison, f)

##### Create TOLNet vs Sonde Regression Dataset

In [6]:
# Create TOLNet vs Sonde Regression Dataset

if (data_dir / 'regression.pkl').is_file():
    with open(data_dir / 'regression.pkl', 'rb') as f:
        regression = pickle.load(f)
else:
    regression = {} 
    for key in comparison.keys(): 
        instrument_key = key[0].split(".")[2]   
        if not instrument_key in regression:
            regression[instrument_key] = []

        temp_tolnet = comparison[key]["right_df"].mean().astype(float)*1000

        temp_tolnet = pd.DataFrame({
            "tolnet_ozone": temp_tolnet.values,
            "tolnet_altitude": temp_tolnet.index.values.astype(float) / 1000
            })

        test_sonde = comparison[key]["left_df"][
            ["Ozone_ppbv", "distance_km", "Altitude_km", "WindSpeed_mps", "WindDirection_deg"]
            ].copy()
        test_sonde.rename(columns={"Ozone_ppbv": "sonde_ozone", "Altitude_km": "sonde_altitude", "distance_km": "sonde_distance"}, inplace=True)

        temp = makePlots._align_heights(test_sonde, temp_tolnet, left_df_column="sonde_altitude", right_df_column="tolnet_altitude", align="right")
        
        if not temp.empty: 
            temp.reset_index(inplace=True)
            temp["tolnet_ozone"] = temp_tolnet["tolnet_ozone"].copy()
            regression[instrument_key].append(temp)

    for key in regression.keys():
        regression[key] = pd.concat(regression[key])
    
    with open(data_dir / "regression.pkl", "wb") as f: 
        pickle.dump(regression, f)

#### Usefull Dictionary Lookups

In [7]:
instrument_names = {
    "csl001_hires_guilford": "NOAA CSL Lidar",
    "gsfc003_hires_oldfield": "NASA GSFC Lidar",
    "gsfc003_hires": "NASA GSFC Lidar",
    'csl001_hires_boulder': "NOAA CSL Lidar",
    "york": "CCNY Lidar",
    "larc001_hires_sherwood": "NASA LaRC Lidar"
}

instrument_locations = {
    "csl001_hires_guilford": "Yale Coastal",
    "gsfc003_hires_oldfield": "Flax Pond",
    "gsfc003_hires": "Flax Pond",
    'csl001_hires_boulder': "Boulder, CO",
    "york": "CCNY Rooftop",
    "larc001_hires_sherwood": "LaRC Sherwood"
}

#### Making Plots

##### Surface Plots

In [21]:
# # airnow, airnow_metadata
for key in airnow_data.keys(): 
    for sub_key in airnow_data[key].keys():
        print(f"{key} - {sub_key}: {len(airnow_data[key][sub_key])} records")

CO - Value: 4966 records
CO - RawConcentration: 4966 records
CO - AQI: 4966 records
CO - Category: 4966 records
CO - geometry: 4966 records
NO2 - Value: 7560 records
NO2 - RawConcentration: 7560 records
NO2 - AQI: 7560 records
NO2 - Category: 7560 records
NO2 - geometry: 7560 records
OZONE - Value: 17896 records
OZONE - RawConcentration: 17896 records
OZONE - AQI: 17896 records
OZONE - Category: 17896 records
OZONE - geometry: 17896 records
PM10 - Value: 5371 records
PM10 - RawConcentration: 5371 records
PM10 - AQI: 5371 records
PM10 - Category: 5371 records
PM10 - geometry: 5371 records
PM2.5 - Value: 21680 records
PM2.5 - RawConcentration: 21680 records
PM2.5 - AQI: 21680 records
PM2.5 - Category: 21680 records
PM2.5 - geometry: 21680 records
SO2 - Value: 5064 records
SO2 - RawConcentration: 5064 records
SO2 - AQI: 5064 records
SO2 - Category: 5064 records
SO2 - geometry: 5064 records


##### Site Maps

In [24]:
tolnet_geometry = []
for key in tolnet.keys(): 
    temp = tolnet[key].geometry.unique()[0]
    tolnet_geometry.append(Point(temp.x, temp.y))
tolnet_geometry = gpd.GeoDataFrame(geometry=tolnet_geometry)

In [26]:
makePlots.site_map(
    instruments={
        "ASOS": asos_geometry,
        "EPA Surface": surface["OZONE"][~surface["OZONE"]['geometry'].duplicated(keep='first')],
        "TOLNet Lidars": tolnet_geometry
        }
    )

##### Lidar Curtains

In [16]:
from collections import defaultdict

for key in tolnet.keys():
    lidar_file = key
    temp_tolnet = tolnet[lidar_file].drop(columns="geometry")

    if "glass" in lidar_file:
        title = f"{instrument_names[lidar_file.split('.')[2]]} GLASS [{temp_tolnet.index.min().strftime('%Y-%m-%d')}]"
    else:
        title = f"{instrument_names[lidar_file.split('.')[2]]} In-House [{temp_tolnet.index.min().strftime('%Y-%m-%d')}]"
    
    lidar = {
        "X": temp_tolnet.index,
        "Y": temp_tolnet.columns.astype(float) / 1000,
        "C": temp_tolnet.values.T * 1000
        }

    date = temp_tolnet.index.min().date()
    savename = title.replace(' ', '_').replace('[', '').replace(']', '').replace(',', '').replace('-', '_') + ".png"
    folder = Path("./figures/curtains_no_sonde") / date.strftime('%Y_%m_%d')
    folder.mkdir(parents=True, exist_ok=True)

    params = {
        "fig.colorbar": {
            "label": "Ozone Mixing Ratio (ppbv)"
            },
        "ax.set_xlim": [date, date + pd.Timedelta(days=1)],
        "ax.set_ylim": [0, 10],
        "ax.set_title": title,
        "fig.savefig": {
            "fname": folder / savename,
            "format": "png",
            "dpi": 600,
            "transparent": True
            }
        }

    makePlots.plot_curtain(lidar, **params)  

In [10]:
run = False

if run==True:
    from collections import defaultdict

    grouped = defaultdict(list)

    for key in comparison.keys():
        grouped[key[0]].append(key)

    for lidar_file, matching_keys in grouped.items():

        temp_sonde = []
        for k in matching_keys:
            temp_sonde.append(comparison[k]["left_df"])

        temp_sondes = pd.concat(temp_sonde)
        temp_tolnet = tolnet[lidar_file].drop(columns="geometry")

        if "glass" in lidar_file:
            title = f"{instrument_names[lidar_file.split('.')[2]]} GLASS [{temp_tolnet.index.min().strftime('%Y-%m-%d')}]"
        else:
            title = f"{instrument_names[lidar_file.split('.')[2]]} In-House [{temp_tolnet.index.min().strftime('%Y-%m-%d')}]"

        sonde = {
            "X": temp_sondes.index.get_level_values('timestamp'),
            "Y": temp_sondes["Altitude_km"],
            "C": temp_sondes["Ozone_ppbv"],
            "vline_params": {
                "colors": 'black', 
                "linewidth": 1.5, 
                "alpha": 0.5, 
                "zorder": 3,
                "linestyle": 'dashed',
                "skip": 15
                }
            }
        
        lidar = {
            "X": temp_tolnet.index,
            "Y": temp_tolnet.columns.astype(float) / 1000,
            "C": temp_tolnet.values.T * 1000
            }

        date = temp_tolnet.index.min().date()
        params = {
            "fig.colorbar": {
                "label": "Ozone Mixing Ratio (ppbv)"
                },
            "ax.set_xlim": [date, date + pd.Timedelta(days=1)],
            "ax.set_ylim": [0, 5],
            "ax.set_title": title,
            "fig.savefig": {
                "fname": Path("./figures/curtains") / f"{title.replace(' ', '_')}.png",
                "format": "png",
                "dpi": 600,
                "transparent": True
                }
            }

        makePlots.plot_curtain(lidar, sonde, **params)  
    # https://nasagov.app.box.com/f/8f4145e4cac8490cbb3dc1fb89c14a29


##### Wind Rose from Sonde

In [11]:
run = False

if run==True:
    for key in regression.keys():
        test = regression[key].dropna()

        name = instrument_locations[key] 

        title = f"{name} Wind Rose from Sonde"

        savename = title.replace(' ', '_').replace('-','_').replace('.', '_') + ".png"

        plot_params = {
            "ax.set_title": {
                "label": title,
                "fontsize": 16,
                "pad": 20,
                },

            "ax.legend": {
                "title": "Wind Speed (m/s)",
                "fontsize": 10,
                "title_fontsize": 10,
                "loc": "center",
                "bbox_to_anchor": (0.5, -0.15),
                "ncols": 3
                },

            "fig.savefig": {
                "fname": Path("./figures/wind_roses") / savename,
                "format": "png",
                "dpi": 600,
                "transparent": True,
                "bbox_inches": "tight"
                },
            }

        makePlots.plot_wind_rose(test['WindDirection_deg'], test['WindSpeed_mps'], plot_params=plot_params)
        plt.close()


##### Plots for Regression (TOLNet vs Sonde)

In [12]:
run = False

if run==True:
    for key in regression.keys():
        test = regression[key].dropna()

        if key == "gsfc003_hires":
            name = f"{instrument_names[key]} (Glass)"
        else:
            name = instrument_names[key] 

        title = f"{name} vs. Ozone-sonde Comparison"

        slope, intercept, r_value, p_value, std_err = linregress(test["sonde_ozone"], test["tolnet_ozone"])

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.scatter(test["sonde_ozone"], test["tolnet_ozone"], c=test["sonde_distance"])
        cbar = fig.colorbar(im, ax=ax, label="Sonde Distance (km)", ticks=np.arange(0, 11, 1))
        cbar.set_label("Sonde Distance (km)")
        cbar.set_ticks(np.arange(0, 11, 1))
        cbar.set_ticklabels([str(i) for i in range(0, 11)])  # explicit labels

        ax.plot(test["sonde_ozone"], slope*test["sonde_ozone"] + intercept, 'r', label='fit')
        ax.set_xlabel("Sonde O3 (ppbv)")
        ax.set_ylabel("Lidar O3 (ppbv)")
        ax.set_xlim([0, 150])
        ax.set_ylim([0, 150])

        eq_text = f"$y = {slope:.2f}x + {intercept:.2f}$\n$R^2 = {r_value**2:.3f}$"
        ax.text(
            0.05, 0.95,
            eq_text,
            transform=ax.transAxes,
            fontsize=12,
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none")
        )
        ax.set_title(title)
        ax.grid({
                "which": 'both',
                "linestyle": '--',
                "linewidth": 0.5,
                "alpha": 0.7
                })

        savename = title.replace(' ', '_').replace('-', '_').replace('.', '_') + ".png"
        plt.savefig(Path("./figures/regressions") / savename, dpi=600, transparent=True)
        plt.close()


##### Vertical Profiles (TOLNet vs Sonde)

In [13]:
run = False

if run==True:
    keys = sorted(list(comparison.keys()))
    check = set()

    for key in keys:
        if key in check: 
            continue
        matching_key = None
        date = comparison[key]["right_df"].index[0].strftime("%Y%m%d")
        filename = key[0]

        profiles = {}

        if "glass" in filename:
            check.add(key)  # avoid matching the same key

            pattern = (
                rf"^groundbased_lidar\.o3_nasa\.gsfc003_hires(?:\.[^.]+)*_oldfield\.ny_"
                rf"{date}t\d{{6}}z_\d{{8}}t\d{{6}}z_\d{{3}}\.h5$"
                )

            matching_key = makePlots.find_matching_key(key, keys, pattern)
            check.add(matching_key)

            profiles["glass"] = makePlots.lidar_XYC(comparison[key]["right_df"].mean()*1000, "profile")
            glass_label = f"{instrument_names[key[0].split('.')[2]]} GLASS"
            profiles["glass"]["params"] = {
                "color": (68/255, 157/255, 209/255),
                "label": glass_label,
                }

        if matching_key: 
            key = matching_key

        profiles["lidar"] = makePlots.lidar_XYC(comparison[key]["right_df"].mean()*1000, "profile")
        lidar_label = f"{instrument_names[key[0].split('.')[2]]} In-House"
        profiles["lidar"]["params"] = {
            "color": (203/255, 121/255, 58/255),
            "label": lidar_label 
            }

        profiles["sonde"] = makePlots.sonde_XYC(comparison[key]["left_df"], "profile")
        sonde_label = f"{key[1].split("_")[0].split("-")[1]} Sonde"

        date = comparison[key]["right_df"].index[0].strftime("%Y%m%dT%H%M%SZ")

        profiles["sonde"]["params"] = {
            "color": (53/255, 53/255, 53/255),
            "label": sonde_label
            }

        savename = f"{lidar_label}_{sonde_label}_{date}.png".replace(" ", "_").replace("-", "_")
        plot_params = {
            "ax.grid": {
                "which": 'both',
                "linestyle": '--',
                "linewidth": 0.5,
                "alpha": 0.7
                },
            "fig.savefig": {
                "fname": Path("./figures/profiles") / savename,
                "format": "png",
                "dpi": 600,
                "transparent": True
                }
            }
        
        makePlots.vertical_profile(profiles, plot_params=plot_params, show=False)
        plt.close()
